In [ ]:
# -*- coding: utf-8 -*-

import os
import json
import re
import time
from pathlib import Path

import pandas as pd
import tiktoken
import httpx
from openai import AzureOpenAI  # AzureOpenAI est bien le client à utiliser côté SDK openai-python 【2-3052f5】【3-5aab42】




SyntaxError: invalid syntax (2417139676.py, line 135)

In [ ]:
# =========================
# 0) Paramètres (sans fonctions inutiles)
# =========================

# Encodage tokens (dans le PDF: o200k_base) 【1-76f1f4】
encoding = tiktoken.get_encoding("o200k_base")

# Endpoint APIM + clé APIM + déploiement (dans le PDF : endpoint APIM + Ocp-Apim-Subscription-Key) 【1-76f1f4】
AZURE_OPENAI_ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip()      # ex: https://...azure-api.net/project-csrd
APIM_SUBSCRIPTION_KEY   = os.getenv("APIM_SUBSCRIPTION_KEY", "").strip()      # subscription key APIM
AZURE_OPENAI_API_KEY    = os.getenv("AZURE_OPENAI_API_KEY", APIM_SUBSCRIPTION_KEY).strip()
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01").strip()
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini").strip()

# Dans le PDF, httpx.Client(verify=False) est utilisé 【1-76f1f4】
# Par défaut on laisse verify=True, mais tu peux forcer verify=False via ALLOW_INSECURE_SSL=1.
ALLOW_INSECURE_SSL = os.getenv("ALLOW_INSECURE_SSL", "0").strip() == "1"
http_client = httpx.Client(verify=not ALLOW_INSECURE_SSL)

if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("AZURE_OPENAI_ENDPOINT manquant.")
if not AZURE_OPENAI_API_KEY:
    raise ValueError("AZURE_OPENAI_API_KEY ou APIM_SUBSCRIPTION_KEY manquant.")
if not AZURE_OPENAI_DEPLOYMENT:
    raise ValueError("AZURE_OPENAI_DEPLOYMENT manquant.")

# Client AzureOpenAI (sans build_client) 【1-76f1f4】【2-3052f5】【3-5aab42】
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    default_headers={"Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY} if APIM_SUBSCRIPTION_KEY else None,
    http_client=http_client
)




In [ ]:
# =========================
# 1) Classification fournisseur (call_chatgpt_1)
# =========================

def call_chatgpt_1(input_text: str) -> str:
    start_time = time.time()

    completion = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0.5,
        messages=[
            {
                "role": "system",
                "content": (
                    "Tu m'aides à savoir si le fournisseur est "
                    "Suez ou Eau Métropole Rouen Normandie ou Veolia "
                    "ou Eau Bordeaux Métropole ou Eau du Grand Lyon "
                    "ou Eau du bassin Rennais. "
                    "Ne réponds que par un seul de ces libellés."
                ),
            },
            {
                "role": "user",
                "content": (
                    "La réponse doit être uniquement : "
                    "Suez ou Eau Métropole Rouen Normandie ou Veolia "
                    "ou Eau Bordeaux Métropole ou Eau du Grand Lyon "
                    "ou Eau du bassin Rennais."
                ),
            },
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[call_chatgpt_1] Nombre de tokens : {len(tokens)}")
    print(f"[call_chatgpt_1] elapsed={time.time() - start_time:.2f}s")

    return (completion.choices[0].message.content or "").strip()




In [ ]:
# =========================
# 2) Scénarios d’extraction (call_chatgpt_scenarioX)
#    Le PDF montre un scénario “Veolia” avec une liste de champs et un format clé:valeur; ... 【1-76f1f4】
# =========================

FIELDS_INSTRUCTIONS = """Extraire exactement les données de cette manière et dans cette ordre et
indique le nom des données demandées et les données présentes dans la facture.
Si l'information demandée n'est pas disponible, indiquer "non disponible"
Ne pas ajouter des commentaires ou des Notes. Séparer chaque donnée avec un point-virgule ;

Contexte (format attendu):
Nom du fournisseur :
Numéro de facture :
Nom du client:
Adresse du client :
Montant facturé :
Période de facturation :
Consommation totale :
Numéro du compteur:
Date d'émission de la facture:
Adresse desservie:
"""



In [ ]:
def call_chatgpt_extract(input_text: str, provider_name: str) -> str:
    """Version factorisée des scénarios 0..4 (au lieu de 5 fonctions quasi identiques)."""
    start_time = time.time()

    completion = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0.5,
        messages=[
            {
                "role": "system",
                "content": (
                    "Vous êtes un assistant utile qui m'aidera à extraire des informations "
                    "du fichier texte qui est le résultat de l'OCR (Azure Document Intelligence avec le modèle de document) "
                    f"d'une facture d'eau de {provider_name}."
                ),
            },
            {
                "role": "user",
                "content": "Pouvez-vous m'aider à extraire les données du fichier de sortie et uniquement les données demandées?",
            },
            {"role": "assistant", "content": FIELDS_INSTRUCTIONS},
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[call_chatgpt_extract:{provider_name}] Nombre de tokens : {len(tokens)}")
    print(f"[call_chatgpt_extract:{provider_name}] elapsed={time.time() - start_time:.2f}s")

    return (completion.choices[0].message.content or "").strip()




In [ ]:
# =========================
# 3) Parsing "clé: valeur; clé: valeur" (parse_pairs)
# =========================

def parse_pairs(answer_text: str) -> dict:
    pairs = [e.strip() for e in str(answer_text).split(";") if e.strip()]
    out = {}
    for pair in pairs:
        if ":" not in pair:
            continue
        k, v = pair.split(":", 1)
        out[k.strip()] = v.strip()
    return out




In [ ]:
# =========================
# 4) Lecture JSON OCR + boucle de traitement
#    Dans le PDF : data['analyzeResult']['content'] sinon data['content'] 【1-76f1f4】
# =========================

def extract_page_number(file_name: str) -> int:
    m = re.search(r"page_(\d+)", file_name)
    return int(m.group(1)) if m else 10**9


def load_content_from_invoice_json(file_path: Path) -> str:
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "analyzeResult" in data and "content" in data["analyzeResult"]:
        return data["analyzeResult"]["content"]
    if isinstance(data, dict) and "content" in data:
        return data["content"]
    return json.dumps(data, ensure_ascii=False)




In [ ]:
def normalize_invoice_number_columns(final_df: pd.DataFrame) -> pd.DataFrame:
    """
    Reproduit l’idée du PDF : combiner 'N de facture' et 'Numéro de facture' puis normaliser le nom. 【1-76f1f4】
    """
    if "Numéro de facture" in final_df.columns:
        final_df["Numéro Facture"] = final_df["Numéro de facture"]
        if "N de facture" in final_df.columns:
            final_df["Numéro Facture"] = final_df["Numéro Facture"].combine_first(final_df["N de facture"])
            final_df = final_df.drop(columns=["N de facture", "Numéro de facture"], errors="ignore")
            final_df = final_df.rename(columns={"Numéro Facture": "Numéro de facture"})
    return final_df




In [ ]:

def main(input_dir: str = "invoices/output") -> pd.DataFrame:
    directory = Path(input_dir)
    file_paths = sorted(
        [p for p in directory.iterdir() if p.name.lower().endswith(".json")],
        key=lambda p: extract_page_number(p.name),
    )

    rows = []

    for file_path in file_paths:
        content = load_content_from_invoice_json(file_path)
        print(f"\n=== {file_path.name} ===")

        provider_answer = call_chatgpt_1(content)
        provider_lower = provider_answer.lower()

        # Routage similaire au PDF (suez/rouen/veolia/bordeaux/lyon/rennes) 【1-76f1f4】
        if "suez" in provider_lower:
            provider = "Suez"
        elif "rouen" in provider_lower:
            provider = "Eau Métropole Rouen Normandie"
        elif "veolia" in provider_lower or "véolia" in provider_lower:
            provider = "Veolia"
        elif "bordeaux" in provider_lower:
            provider = "Eau Bordeaux Métropole"
        elif "lyon" in provider_lower:
            provider = "Eau du Grand Lyon"
        elif "rennes" in provider_lower or "bassin" in provider_lower:
            provider = "Eau du bassin Rennais"
        else:
            provider = provider_answer.strip() or "non disponible"

        answer = call_chatgpt_extract(content, provider)
        data = parse_pairs(answer)

        # Métadonnées utiles
        data["__source_file"] = file_path.name
        data["__provider_classified"] = provider

        rows.append(data)

    final_df = pd.DataFrame(rows)
    final_df = normalize_invoice_number_columns(final_df)

    return final_df


if __name__ == "__main__":
    main()

In [ ]:
import time
import tiktoken

def agent_correcteur(input_text, colonne_recherchee, client):
    """
    Agent générique: récupère UNE info précise (colonne_recherchee) et demande de la renvoyer entre ** **.
    Visible dans 1.pdf. 【1-7ba7fb】
    """
    start_time = time.time()
    encoding = tiktoken.encoding_for_model("gpt-40-mini")  # comme dans le PDF 【1-7ba7fb】

    completion = client.chat.completions.create(
        model="gpt-40-mini",
        temperature=0.8,
        messages=[
            {"role": "system", "content": "Tu es un agent qui récupère une information précise et seulement une information."},
            {"role": "user", "content": f"Donne la valeur de la colonne {colonne_recherchee} entre ** **"},
            {"role": "assistant", "content": f"Donne la valeur de la colonne {colonne_recherchee} entre ** **"},
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[agent_correcteur] tokens={len(tokens)} elapsed={time.time()-start_time:.2f}s")
    return (completion.choices[0].message.content or "").strip()









In [ ]:
def agent_correcteur_bis(input_text, client):
    """
    Agent spécialisé: Adresse desservie.
    Visible dans 1.pdf (agent_correcteur_bis). 【1-7ba7fb】
    """
    start_time = time.time()
    encoding = tiktoken.encoding_for_model("gpt-40-mini")

    completion = client.chat.completions.create(
        model="gpt-40-mini",
        temperature=0.8,
        messages=[
            {"role": "system", "content": "Tu es un agent qui récupère une information précise l'adresse desservie et seulement une information."},
            {"role": "user", "content": "Donne l'adresse desservie entre ** **"},
            {"role": "assistant", "content": "Donne l'adresse desservie entre ** **"},
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[agent_correcteur_bis] tokens={len(tokens)} elapsed={time.time()-start_time:.2f}s")
    return (completion.choices[0].message.content or "").strip()

In [ ]:
def agent_correcteur_bis1(input_text, client):

    start_time = time.time()
    encoding = tiktoken.encoding_for_model("gpt-40-mini")

    completion = client.chat.completions.create(
        model="gpt-40-mini",
        temperature=0.8,
        messages=[
            {"role": "system", "content": "Tu es un agent qui récupère une information précise la période de facturation et seulement une information."},
            {"role": "user", "content": "Donne la période de facturation entre ** **"},
            {"role": "assistant", "content": "Donne la période de facturation entre ** **"},
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[agent_correcteur_bis1] tokens={len(tokens)} elapsed={time.time()-start_time:.2f}s")
    return (completion.choices[0].message.content or "").strip()


In [ ]:
def agent_correcteur_bis2(input_text, client):
   
    start_time = time.time()
    encoding = tiktoken.encoding_for_model("gpt-40-mini")

    completion = client.chat.completions.create(
        model="gpt-40-mini",
        temperature=0.8,
        messages=[
            {"role": "system", "content": "Tu es un agent qui récupère une information précise le numéro du compteur et seulement une information."},
            {"role": "user", "content": "Donne le numéro du compteur entre ** **"},
            {"role": "assistant", "content": "Donne le numéro du compteur entre ** **"},
            {"role": "user", "content": input_text},
        ],
    )

    tokens = encoding.encode(input_text)
    print(f"[agent_correcteur_bis2] tokens={len(tokens)} elapsed={time.time()-start_time:.2f}s")
    return (completion.choices[0].message.content or "").strip()

In [ ]:
import os
import json
import re

# Regex unique pour extraire le texte entre ** **
PATTERN_BETWEEN_STARS = re.compile(r"\*\*(.*?)\*\*", re.DOTALL)

def extract_between_stars(text: str):
    """Retourne le contenu entre ** **, ou None."""
    m = PATTERN_BETWEEN_STARS.search(text or "")
    return m.group(1).strip() if m else None

def ask_until_match(ask_fn, max_retry=10):
    """
    ask_fn() -> str
    On repose la question jusqu'à obtenir **...** ou atteindre max_retry.
    (Dans 10.pdf, c'est le while t_out < 10 répété partout.) 【1-e7027d】
    """
    for attempt in range(1, max_retry + 1):
        raw = ask_fn()
        extracted = extract_between_stars(raw)
        if extracted:
            return extracted, raw, attempt
        print(f"Retry {attempt}/{max_retry} : pas de ** ** détecté, on repose la question.")
    return "non disponible", raw, max_retry


# Mapping colonne -> fonction d'appel agent
# (Dans 10.pdf: Adresse desservie => agent_correcteur_bis ; Période => bis1 ; Compteur => bis2 ; sinon agent_correcteur) 【1-e7027d】
AGENT_BY_COLUMN = {
    "Adresse desservie": lambda content, col: agent_correcteur_bis(content, client),
    "Période de facturation": lambda content, col: agent_correcteur_bis1(content, client),
    "Numéro du compteur": lambda content, col: agent_correcteur_bis2(content, client),
}

def correct_non_disponible(final_df, file_names, null_info, workout_dir):
    """
    - final_df : DataFrame résultat initial
    - file_names : liste des fichiers JSON OCR (alignés avec l'index df)
    - null_info : liste de tuples (col, idx, val) où val == 'non disponible'
      (vu dans 10.pdf: null_info contient des éléments du type ('Consommation totale', idx, 'non disponible')) 【1-e7027d】
    - workout_dir : dossier contenant les JSON
    """

    # Petit gain: évite file_names.index(file_name) dans une boucle (coût inutile)
    for idx_file, file_name in enumerate(file_names):
        file_path = os.path.join(workout_dir, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Dans 10.pdf: analyzeResult/content sinon content 【1-e7027d】
        content = data["analyzeResult"]["content"] if "analyzeResult" in data else data.get("content", "")

        # On ne traite que les colonnes "non disponible" de CET index
        # => au lieu de parcourir tout null_info pour rien, on filtre.
        targets = [(col, idx, val) for (col, idx, val) in null_info if idx == idx_file]

        if not targets:
            continue

        for col, idx, val in targets:
            # Choix agent: spécialisé si colonne connue, sinon générique
            agent_caller = AGENT_BY_COLUMN.get(
                col,
                lambda content, col: agent_correcteur(content, col, client)
            )

            # Fonction "ask" adaptée (certains agents n'ont pas besoin de col)
            extracted_value, raw_answer, tries = ask_until_match(
                ask_fn=lambda: agent_caller(content, col),
                max_retry=10
            )

            print(f"✅ Fichier={file_name} | Index={idx} | Colonne={col} | Tries={tries} | Valeur={extracted_value}")

            final_df.at[idx, col] = extracted_value

    return final_df